In [8]:
!pip install -q librosa numpy

import os
import json
import numpy as np 
import librosa # For audio processing
from pathlib import Path
from typing import Tuple, List, Dict
import pickle # For saving and loading data

In [27]:
class AudoPreprocessor:
    # 30s audio clips at 16kHz
    def __init__(self, data_dir, output_dir, epoch_duration = 30, sample_rate = 16000):
        self.data_dir = Path(data_dir)
        self.output_dir = Path(output_dir)
        self.epoch_duration = epoch_duration
        self.sample_rate = sample_rate
        
        # AST Input Shape
        self.target_length = sample_rate * epoch_duration
        
        # Make output directory if it doesn't exist
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
        print(f"\nInitialized AudoPreprocessor")
        print(f"    Data_dir: {data_dir}")
        print(f"    Output_dir: {output_dir}")
        print(f"    Epoch Duration: {epoch_duration}s")
        print(f"    Sample Rate: {sample_rate}Hz")
        print(f"    Samples per epoch: {self.target_length}")
        
    def load_annotations(self, folder_id) -> Dict:
        # load annotations from json like 01_annotation.json
        annotation_path = self.data_dir / folder_id / f"{folder_id}_annotation.json"
        
        if (not annotation_path.exists()):
            raise FileNotFoundError(f"Annotation file not found: {annotation_path}")
        
        with open(annotation_path, 'r') as f:
            annotations = json.load(f)
            
        # Debug print statements
        print(f"\nLoaded annotations from {annotation_path}")
        print(f"    Record Start: {annotations['record_start']}s")
        print(f"    Awake Intervals: {len(annotations['awake_intervals'])}")
        print(f"    Events: {len(annotations['events'])}")
        
        return annotations
    
    def load_audio(self, folder_id) -> Tuple[np.ndarray, int]:
        # load audio file like 01_phone.wav
        audio_path = self.data_dir / folder_id / f"{folder_id}_phone.wav"
        
        if (not audio_path.exists()):
            raise FileNotFoundError(f"Audio file not found: {audio_path}")
        
        # Librosa audio loading
        audio, sr = librosa.load(audio_path, sr=self.sample_rate, mono=True)
        
        # Debug print statements
        print(f"\nLoaded audio from {audio_path}")
        print(f"    Audio Shape: {audio.shape}")
        print(f"    Sample Rate: {sr}Hz")
        print(f"    Duration: {len(audio)/sr:.2f}s")
        
        return audio, sr
    
    # Check if time point is within any awake interval
    def is_awake(self, time_point: float, awake_intervals: List[Tuple[float]]) -> bool:
        for start, end in awake_intervals:
            if start <= time_point <= end:
                return True
        return False
        
    # Extract epoch labels based on annotations
    def extract_epoch_labels(self, epoch_start: float, epoch_end: float, events: List[Dict], awake_intervals: List[List[float]]) -> int:
        # Check if epoch during awake interval
        if (self.is_awake(epoch_start, awake_intervals) or self.is_awake(epoch_end, awake_intervals)):
            return -1  # Awake
        
        label = 0 # Default to no event (1 = osa [obstructive sleep apnea], 2 = hyp [hypnopnea])
        
        # Gonna prioritize in order hypo > osa > none
        for event in events:
            event_start = event['evnet_start']  # Note: typo in original data
            event_end = event_start + event['event_duration']
            event_type = event['event_type']
            
            # Check if event overlaps with epoch
            if not (event_end < epoch_start or event_start > epoch_end):
                if event_type == 'hypo':
                    label = max(label, 2)
                elif event_type == 'osa':
                    label = max(label, 1)
        
        return label
    
    # Create epochs from audio data and label them
    def create_epochs(self, folder_id: str) -> Tuple[List[np.ndarray], List[int]]:
        # Load data
        annotations = self.load_annotations(folder_id)
        audio, sr = self.load_audio(folder_id)
        
        # Extract Annotations
        record_start = annotations['record_start']
        awake_intervals = annotations['awake_intervals']
        events = annotations['events']
        
        # Number of epochs
        audio_duration = len(audio) / sr
        num_epochs = int(np.floor(audio_duration / self.epoch_duration))
        
        print (f"\nCreating {num_epochs} epochs of {self.epoch_duration}s each from audio of duration {audio_duration:.2f}s")
        
        epochs = []
        labels = []
        label_counts = { -1: 0, 0: 0, 1: 0, 2: 0 } # Awake, No Event, OSA, Hypo
        
        for i in range(num_epochs):
            epoch_start_sample = i * self.target_length
            epoch_end_sample = (i + 1) * self.target_length
            
            # Handle last epoch case if it too short
            if epoch_end_sample > len(audio):
                break
            
            epoch_audio = audio[epoch_start_sample:epoch_end_sample]
            
            # Actual start and end time per recording start
            epoch_start_time = record_start + (i * self.epoch_duration)
            epoch_end_time = epoch_start_time + self.epoch_duration
            
            label = self.extract_epoch_labels(epoch_start_time, epoch_end_time, events, awake_intervals)
            
            # Don't care if awake
            if label == -1:
                label_counts[-1] += 1
                continue
            
            epochs.append(epoch_audio)
            labels.append(label)
            label_counts[label] += 1
            
        print(f"\nEpoch Statistics for folder {folder_id}:")
        print(f"    Total Epochs: {num_epochs}")
        print(f"    Processed Epochs Saved: {len(epochs)}")
        print(f"    Awake Epochs Skipped: {label_counts[-1]}")
        print(f"    No Event Epochs: {label_counts[0]}")
        print(f"    OSA Event Epochs: {label_counts[1]}")
        print(f"    Hypopnea Event Epochs: {label_counts[2]}")
        
        return epochs, labels
    
    def process_folders(self, folder_ids: List[str] = None):
        
        # If no folder IDs provided, process all folders in data_dir
        if (not folder_ids):
            print ("\nNo folder IDs provided. Defaulting to folders 01-50.")
            folder_ids = [f"{i:02d}" for i in range(1, 51)] # Folders named 01 to 50
        
        all_epochs = []
        all_labels = []
        all_folder_ids = []
        
        print(f"\n{'='*40}")
        print(f"Processing folders {len(folder_ids)} folders")
        print(f"{'='*40}")
        
        for folder_id in folder_ids:
            print(f"\nProcessing folder {folder_id}...")
            
            epochs, labels = self.create_epochs(folder_id)
            
            all_epochs.extend(epochs)
            all_labels.extend(labels)
            all_folder_ids.extend([folder_id] * len(epochs))
            
            print(f"Completed processing folder {folder_id}. Total epochs so far: {len(all_epochs)}")
            
        # Save all processed data
        print(f"\n{'='*40}")
        print(f"Saving preprocessed data...")
        print(f"{'='*40}")
        
        data = {
            'epochs': np.array(all_epochs),
            'labels': np.array(all_labels),
            'folder_ids': all_folder_ids,
            'sample_rate': self.sample_rate,
            'epoch_duration': self.epoch_duration
        }
        
        output_path = self.output_dir / "preprocessed_data.pkl"
        with open(output_path, 'wb') as f:
            pickle.dump(data, f)
            
        print(f"\nPreprocessed data saved to {output_path}")
        print(f"    Total Epochs Saved: {len(all_epochs)}")
        print(f"    No-event Epochs: {all_labels.count(0)}")
        print(f"    OSA Event Epochs: {all_labels.count(1)}")
        print(f"    Hypopnea Event Epochs: {all_labels.count(2)}")
        print(f"    Data Shape: {data['epochs'].shape}")
        
        return data

if __name__ == "__main__":
    parent_dir = os.path.dirname(os.getcwd())
    
    # Define paths relative to the current directory
    DATA_DIR = parent_dir + "/Data"
    OUTPUT_DIR = parent_dir + "/Preprocessed"
    
    preprocessor = AudoPreprocessor(
        data_dir=DATA_DIR,
        output_dir=OUTPUT_DIR,
        epoch_duration=30,
        sample_rate=16000
    )
    
    test_folder_ids = [f"{i:02d}" for i in range(1, 3)]
    
    data = preprocessor.process_folders(folder_ids=test_folder_ids)


Initialized AudoPreprocessor
    Data_dir: c:\Users\jacst\Downloads\School\Fall25\CSE575/Data
    Output_dir: c:\Users\jacst\Downloads\School\Fall25\CSE575/Preprocessed
    Epoch Duration: 30s
    Sample Rate: 16000Hz
    Samples per epoch: 480000

Processing folders 2 folders

Processing folder 01...

Loaded annotations from c:\Users\jacst\Downloads\School\Fall25\CSE575\Data\01\01_annotation.json
    Record Start: 75454.0s
    Awake Intervals: 23
    Events: 86

Loaded audio from c:\Users\jacst\Downloads\School\Fall25\CSE575\Data\01\01_phone.wav
    Audio Shape: (356126720,)
    Sample Rate: 16000Hz
    Duration: 22257.92s

Creating 741 epochs of 30s each from audio of duration 22257.92s

Epoch Statistics for folder 01:
    Total Epochs: 741
    Processed Epochs Saved: 676
    Awake Epochs Skipped: 65
    No Event Epochs: 566
    OSA Event Epochs: 1
    Hypopnea Event Epochs: 109
Completed processing folder 01. Total epochs so far: 676

Processing folder 02...

Loaded annotations fro